In [2]:
import pandas as pd
import sys
import os
import numpy as np
import scanpy as sc

In [3]:
adata = sc.read_h5ad(r"D:\Trapecar\250307_gut_liver_blood_ultimate_annotated.h5ad")

In [4]:
adata_TCR = adata[adata.obs['chain_pairing'].isin(["single pair", "extra VJ","extra VDJ","two full chains"]),:]
adata_ab = adata_TCR[adata_TCR.obs['general type'].isin(['TCRab CD4','TCRab CD8aa','TCRab CD8ab']),:]
adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)
adata_ab.obs['subject:condition']= adata_ab.obs['Donor ID'].astype(str).map(str) + ':' + adata_ab.obs['tissue+celltype'].astype(str).map(str)

C:\Users\andre\AppData\Local\Temp\ipykernel_52040\3865754011.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)


In [5]:
# from conga/tcrdist
clone_counts= pd.read_csv(r"G:\My Drive\result\publication\cellreport\revision\conga\gut_liver_TRM_clones_with0.tsv",sep = '\t',index_col = 0)
temp_dict_df = clone_counts[clone_counts['clone_size']>0][['clone_id','va_gene','vb_gene','cdr3a','cdr3b']]
temp_dict_df['clone_code'] = temp_dict_df['va_gene'].astype(str).map(str) + ' ' + temp_dict_df['vb_gene'].astype(str).map(str) + ' ' + temp_dict_df['cdr3a'].astype(str).map(str) + ' ' + temp_dict_df['cdr3b'].astype(str).map(str)
clone_replace_dict = temp_dict_df[['clone_id','clone_code']].set_index('clone_code').sort_index()['clone_id'].to_dict()

In [6]:
os.chdir(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_input")
names = ['TCRab CD4','TCRab CD8ab']
for i in names:
    adata_slice = adata_ab[adata_ab.obs['general type'] == i,:]
    if i == 'TCRab CD8ab':
        adata_slice = adata_slice[
            (adata_slice.obs['tissue+celltype'] != 'L TCRab CD8ab MAIT') |
            ((adata_slice.obs['tissue+celltype'] == 'L TCRab CD8ab MAIT') & (adata_slice.obs['TRAV'] == 'TRAV1-2')), # keep only MAIT with canonical TCR, or the result will be really confusing
            :
        ]

    clone_df = adata_slice.obs[['subject:condition','TRAV', 'TRBV','TRBJ', 'cdr3a', 'cdr3b','clone_code']]
    clone_counts = clone_df.groupby(['subject:condition','clone_code']).size().reset_index(name='clone frequency')
    clone_counts[['TRAV','TRBV','cdr3a','cdr3b']] = clone_counts['clone_code'].str.split(' ', expand=True)
    
    Jmap = clone_df[['TRBJ','clone_code']].drop_duplicates()
    Jmap = Jmap.set_index('clone_code')
    Jmap_bdict = Jmap['TRBJ'].to_dict()
    clone_counts['TRBJ'] =clone_counts['clone_code'].replace(Jmap_bdict)

    clone_counts = clone_counts[clone_counts['clone frequency'] >= 1]
    clone_counts['clone_id'] = clone_counts['clone_code'].map(clone_replace_dict)
    clone_counts[['cdr3b','TRBV','TRBJ','cdr3a','subject:condition','clone frequency']].to_csv(i+'_GLIPH2.tsv', sep="\t",index = False, header = False)

### Mapping GLIPH2 reulst back

#### CD4

In [41]:
CD4_GLIPH2 = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD4_GLIPH2.csv",index_col = 0).iloc[:,0:18]

In [42]:
CD4_GLIPH2['celltype'] = CD4_GLIPH2['Sample'].str.split(':').str[1]
CD4_GLIPH2['donor']    = CD4_GLIPH2['Sample'].str.split(':').str[0]
CD4_GLIPH2['clone'] = (
    CD4_GLIPH2
    .groupby(['donor','TcRb','V','J','TcRa'], dropna=False, sort=False)
    .ngroup()
)
motif_col = 'type' # 'motif' or 'global'
dedup = (
    CD4_GLIPH2
    .loc[:, ['donor','celltype',motif_col,'clone']]
    .drop_duplicates()
)

In [29]:
dedup

,donor,celltype,type,clone
index,,,,
1,Donor AJD3280,LP TCRab CD4 TRM,motif-HWNT motif-HWN,0
1,Donor AJD3280,PB TCRab CD4 Naive/TCM,motif-HWNT motif-HWN,1
1,Donor AJD3280,LP TCRab CD4 Mobile TRM,motif-HWNT motif-HWN,2
1,Donor AJD3280,LP TCRab CD4 TRM,motif-HWNT motif-HWN,3
1,Donor AJKQ118,PB TCRab CD4 Naive/TCM,motif-HWNT motif-HWN,4
...,...,...,...,...
11801,Donor AJD3280,LP TCRab CD4 TRM,global-RRSGWQET,2981
11838,Donor AJG2309,L TCRab CD4 TCM,global-SLAVGTRSRAE,2982
11838,Donor AJG2309,PB TCRab CD4 Naive/TCM,global-SLAVGTRSRAE,2982


In [43]:
CD4_GLIPH2_donor_clone_normalized = {}

for i in dedup['donor'].unique():
    df_i = dedup[dedup['donor'] == i]

    nodes = sorted(df_i['celltype'].unique())
    shared_counts = pd.DataFrame(0.0, index=nodes, columns=nodes, dtype=float)

    adata_temp = adata_ab[adata_ab.obs['Donor ID'] == i]
    adata_temp = adata_temp[adata_temp.obs['general type'] == 'TCRab CD4']
    unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()

    # Precompute sets for speed
    clones_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, 'clone']) for ct in nodes}
    motifs_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, motif_col]) for ct in nodes}

    for A in nodes:
        denom = unique_clone_counts.loc[A]
        if denom == 0:
            continue
        for B in nodes:
            if A == B:
                continue
            # Motifs present in B
            motifs_B = motifs_in_ct[B]
            if not motifs_B:
                shared_counts.loc[A, B] = 0.0
                continue

            # Clones in A that have at least one motif also seen in B
            clones_A_shared = set(
                df_i.loc[
                    (df_i['celltype']==A) & (df_i[motif_col].isin(motifs_B)), #well that is, if a clone in cell type A and has the motif that is in cell type B
                    'clone'
                ]
            )
            shared_counts.loc[A, B] = len(clones_A_shared) / denom
            if shared_counts.loc[A, B] < 2/ denom:
                shared_counts.loc[A, B] = 0 # filter out too few shared

    CD4_GLIPH2_donor_clone_normalized[i] = shared_counts

# --- 2) Average across donors (aligning all cell types) ---
all_nodes = sorted(dedup['celltype'].unique())
dfs = [m.reindex(index=all_nodes, columns=all_nodes, fill_value=0.0) for m in CD4_GLIPH2_donor_clone_normalized.values()]
dfs_array = np.array([df.values.astype(float) for df in dfs])
nonzero_mask = np.sum(dfs_array != 0, axis=0) >= 2
mean_matrix = np.zeros_like(dfs_array[0])

mean_matrix[nonzero_mask] = np.mean(dfs_array[:, nonzero_mask], axis=0)
CD4_GLIPH2_donor_clone_normalized_mean = pd.DataFrame(mean_matrix, index=all_nodes, columns=all_nodes)
CD4_GLIPH2_donor_clone_normalized_mean

C:\Users\andre\AppData\Local\Temp\ipykernel_52040\3680244688.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
C:\Users\andre\AppData\Local\Temp\ipykernel_52040\3680244688.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
C:\Users\andre\AppData\Local\Temp\ipykernel_52040\3680244688.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to

,IEL TCRab CD4 FOXP3+ Treg,IEL TCRab CD4 Mobile TRM,IEL TCRab CD4 TRM,L TCRab CD4 FOXP3+ Treg,L TCRab CD4 Naive/TCM,L TCRab CD4 TCM,L TCRab CD4 TRM,LP TCRab CD4 FOXP3+ Treg,LP TCRab CD4 Mobile TRM,LP TCRab CD4 Naive/TCM,LP TCRab CD4 Poised TCM,LP TCRab CD4 TRM,LP TCRab CD4 Tph,PB TCRab CD4 FOXP3+ Treg,PB TCRab CD4 Naive/TCM,PB TCRab CD4 TCM
IEL TCRab CD4 FOXP3+ Treg,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.218725,0.000000,0.0,0.000000,0.068837,0.000000,0.000000,0.064961,0.0
IEL TCRab CD4 Mobile TRM,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.060317,0.000000,0.069841,0.0,0.000000,0.084656,0.000000,0.000000,0.031746,0.0
IEL TCRab CD4 TRM,0.000000,0.000000,0.000000,0.0,0.017799,0.018749,0.088413,0.022868,0.000000,0.0,0.000000,0.221983,0.041799,0.013516,0.047215,0.0
L TCRab CD4 FOXP3+ Treg,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
L TCRab CD4 Naive/TCM,0.000000,0.000000,0.037473,0.0,0.000000,0.000000,0.123747,0.000000,0.000000,0.0,0.000000,0.059695,0.000000,0.000000,0.038562,0.0
L TCRab CD4 TCM,0.000000,0.000000,0.038718,0.0,0.000000,0.000000,0.165456,0.000000,0.000000,0.0,0.000000,0.061814,0.000000,0.044570,0.093239,0.0
L TCRab CD4 TRM,0.000000,0.005336,0.032590,0.0,0.015144,0.033427,0.000000,0.000000,0.024863,0.0,0.000000,0.112578,0.005043,0.045645,0.062385,0.0
LP TCRab CD4 FOXP3+ Treg,0.053779,0.000000,0.012106,0.0,0.000000,0.000000,0.000000,0.000000,0.008557,0.0,0.000000,0.032397,0.000000,0.015609,0.038711,0.0
LP TCRab CD4 Mobile TRM,0.000000,0.033427,0.000000,0.0,0.000000,0.000000,0.061967,0.007696,0.000000,0.0,0.005995,0.098504,0.000000,0.015252,0.040380,0.0
LP TCRab CD4 Naive/TCM,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [44]:
CD4_GLIPH2_donor_clone_normalized_mean.to_csv("C:/Users/andre/Documents/GitHub/gut-liver-TRM/Revision/GLIPH2/CD4_GLIPH2_donor_separated_motif_shared_normalized_mean.csv")

#### now do the same for CD8ab

In [45]:
CD8_GLIPH2 = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD8_GLIPH2.csv",index_col = 0).iloc[:,0:18]
CD8_GLIPH2['celltype'] = CD8_GLIPH2['Sample'].str.split(':').str[1]
CD8_GLIPH2['donor']    = CD8_GLIPH2['Sample'].str.split(':').str[0]
CD8_GLIPH2['clone'] = (
    CD8_GLIPH2
    .groupby(['donor','TcRb','V','J','TcRa'], dropna=False, sort=False)
    .ngroup()
)
motif_col = 'type'
dedup = (
    CD8_GLIPH2
    .loc[:, ['donor','celltype',motif_col,'clone']]
    .drop_duplicates()
)
CD8_GLIPH2_donor_clone_normalized = {}

for i in dedup['donor'].unique():
    df_i = dedup[dedup['donor'] == i]

    nodes = sorted(df_i['celltype'].unique())
    shared_counts = pd.DataFrame(0.0, index=nodes, columns=nodes, dtype=float)

    adata_temp = adata_ab[adata_ab.obs['Donor ID'] == i]
    adata_temp = adata_temp[adata_temp.obs['general type'] == 'TCRab CD8ab']
    adata_temp = adata_temp[
            (adata_temp.obs['tissue+celltype'] != 'L TCRab CD8ab MAIT') |
            ((adata_temp.obs['tissue+celltype'] == 'L TCRab CD8ab MAIT') & (adata_temp.obs['TRAV'] == 'TRAV1-2')),
            :
        ]
    unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
    
    # Precompute sets for speed
    clones_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, 'clone']) for ct in nodes}
    motifs_in_ct = {ct: set(df_i.loc[df_i['celltype']==ct, motif_col]) for ct in nodes}

    for A in nodes:
        denom = unique_clone_counts[A]
        if denom == 0:
            continue
        for B in nodes:
            if A == B:
                continue
            # Motifs present in B
            motifs_B = motifs_in_ct[B]
            if not motifs_B:
                shared_counts.loc[A, B] = 0.0
                continue

            # Clones in A that have at least one motif also seen in B
            clones_A_shared = set(
                df_i.loc[
                    (df_i['celltype']==A) & (df_i[motif_col].isin(motifs_B)), #well that is, if a clone in cell type A and has the motif that is in cell type B
                    'clone'
                ]
            )
            shared_counts.loc[A, B] = len(clones_A_shared) / denom
            if shared_counts.loc[A, B] < 2/ denom:
                shared_counts.loc[A, B] = 0

    CD8_GLIPH2_donor_clone_normalized[i] = shared_counts

all_nodes = sorted(dedup['celltype'].unique())
dfs = [m.reindex(index=all_nodes, columns=all_nodes, fill_value=0.0) for m in CD8_GLIPH2_donor_clone_normalized.values()]
dfs_array = np.array([df.values.astype(float) for df in dfs])
nonzero_mask = np.sum(dfs_array != 0, axis=0) >= 2
mean_matrix = np.zeros_like(dfs_array[0])

mean_matrix[nonzero_mask] = np.mean(dfs_array[:, nonzero_mask], axis=0)
CD8_GLIPH2_donor_clone_normalized_mean = pd.DataFrame(mean_matrix, index=all_nodes, columns=all_nodes)
CD8_GLIPH2_donor_clone_normalized_mean.to_csv("C:/Users/andre/Documents/GitHub/gut-liver-TRM/Revision/GLIPH2/CD8_GLIPH2_donor_separated_motif_shared_normalized_mean.csv")

C:\Users\andre\AppData\Local\Temp\ipykernel_52040\904399068.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
C:\Users\andre\AppData\Local\Temp\ipykernel_52040\904399068.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_clone_counts = adata_temp.obs.groupby(['tissue+celltype'])['clone_code'].nunique()
C:\Users\andre\AppData\Local\Temp\ipykernel_52040\904399068.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to re

In [46]:
CD8_GLIPH2_donor_clone_normalized_mean

,IEL TCRab CD8ab TRM,L TCRab CD8ab MAIT,L TCRab CD8ab Naive/TCM,L TCRab CD8ab TRM,L TCRab CD8ab Teff,LP TCRab CD8ab TCM,LP TCRab CD8ab TEM,LP TCRab CD8ab TRM,PB TCRab CD8ab Naive/TCM,PB TCRab CD8ab TEM,PB TCRab CD8ab Teff
IEL TCRab CD8ab TRM,0.000000,0.003163,0.016644,0.069901,0.019351,0.004450,0.055651,0.098228,0.019927,0.011029,0.023416
L TCRab CD8ab MAIT,0.000000,0.000000,0.000000,0.403846,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
L TCRab CD8ab Naive/TCM,0.146448,0.000000,0.000000,0.240786,0.034091,0.000000,0.063447,0.048321,0.038826,0.000000,0.105626
L TCRab CD8ab TRM,0.107930,0.021570,0.036344,0.000000,0.078563,0.011388,0.069151,0.030276,0.024326,0.048886,0.130037
L TCRab CD8ab Teff,0.152437,0.000000,0.027420,0.315660,0.000000,0.016634,0.164392,0.000000,0.016634,0.093307,0.453411
LP TCRab CD8ab TCM,0.183245,0.000000,0.000000,0.151194,0.000000,0.000000,0.000000,0.000000,0.000000,0.183761,0.176835
LP TCRab CD8ab TEM,0.248580,0.000000,0.024368,0.119832,0.099900,0.000000,0.000000,0.110944,0.000000,0.059115,0.129643
LP TCRab CD8ab TRM,0.445786,0.000000,0.018229,0.086575,0.000000,0.000000,0.118069,0.000000,0.043050,0.025296,0.019668
PB TCRab CD8ab Naive/TCM,0.050382,0.000000,0.015249,0.031470,0.003662,0.000000,0.000000,0.016846,0.000000,0.012559,0.013132
PB TCRab CD8ab TEM,0.114090,0.000000,0.000000,0.179480,0.111182,0.049054,0.133664,0.058629,0.045296,0.000000,0.193830
